In [1]:
import os
import torchvision.models as models
from PIL import Image
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset
from torchvision.datasets.utils import download_and_extract_archive

from torch.utils.data import DataLoader
from torchvision import transforms

# Fine-tuning and bounding boxes
- we take pre-trained model (e.g. ResNet) on image classification task
- then we fine-tune the weights on bounding box task
- we use Penn-Fudan database (see https://www.cis.upenn.edu/~jshi/ped_html/)
- finally, we evaluate the model performance using standard metric
- TODOs: 1) Implement ResNetBoxes, 2) Choose loss function, 3) Implement IoU as a metric and evaluate performance, 4) Improve ResNetBoxes with argument that determines if the backbone network should be freezed for fine-tuning, 5) Observe performance when different ResNet variants are used

In [2]:
try: # disable certificate verification, needed on MacOS
    import ssl
    ssl._create_default_https_context = ssl._create_unverified_context
except ImportError:
    pass  # SSL module not available, skipping workaround

In [3]:
# Download and extract dataset
url = "https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip"
download_and_extract_archive(url, download_root="data/", extract_root="data/", remove_finished=True)

100%|██████████| 53.7M/53.7M [00:10<00:00, 5.25MB/s]


Extracting data/PennFudanPed.zip to data/


In [4]:
class PennFudanDataset(Dataset):
    def __init__(self, root, transforms=None):
        self.root = root
        self.transforms = transforms
        self.img_dir = os.path.join(root, "PNGImages")  
        self.mask_dir = os.path.join(root, "PedMasks")
        self.imgs = sorted(os.listdir(self.img_dir))
        self.masks = sorted(os.listdir(self.mask_dir))

    def __getitem__(self, idx):
        # Construct full paths
        img_path = os.path.join(self.img_dir, self.imgs[idx])
        mask_path = os.path.join(self.mask_dir, self.masks[idx])

        # Load image and mask
        img = Image.open(img_path).convert("RGB")
        mask = np.array(Image.open(mask_path))

        # Remove background (assumed to be ID 0)
        obj_ids = np.unique(mask)
        obj_ids = obj_ids[obj_ids != 0]

        # Generate binary masks
        masks = mask == obj_ids[:, None, None]

        # Generate bounding boxes
        boxes = []
        for m in masks:
            pos = np.where(m)
            xmin, xmax = pos[1].min(), pos[1].max()
            ymin, ymax = pos[0].min(), pos[0].max()
            boxes.append([xmin, ymin, xmax, ymax])
        boxes = torch.tensor(boxes, dtype=torch.float32)

        # All objects are labeled as class 1 (pedestrian)
        labels = torch.ones((len(obj_ids),), dtype=torch.int64)

        # Construct target dictionary
        target = {
            "box": boxes[0],  # First object's box only
            "label": labels[0],
        }

        if self.transforms:
            img = self.transforms(img)

        return img, target

    def __len__(self):
        return len(self.imgs)

In [5]:
class ResNetBoxes(nn.Module):
    def __init__(self, resnet, freeze=False):
        super().__init__()
        # TODO 1: Use the backbone from `resnet` (all layers except the final FC).
        #         Add adaptive average pooling, a flatten layer, and a small head
        #         (e.g. Linear -> ReLU -> Linear) that outputs 4 bounding box coordinates.
        # Hint: resnet.fc.in_features gives the backbone's output channel count.

        self.backbone = nn.Sequential(*list(resnet.children())[:-2]) #Two things worth noting about the [:-2] slice here — since you're removing ResNet's own avgpool and fc together, your AdaptiveAvgPool2d fully replaces it, which is correct. If you used [:-1] (dropping only fc), ResNet's built-in pool would already reduce spatial dims to 1×1, making your pool redundant (harmless but unnecessary).
        num_features = resnet.fc.in_features #num_features je počet dimenzí na "výstupu" toho předtrénovaného modelu, kterému jsme uřízli poslední vrstvu a místo ní tam budeme dávat tu hlavu

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.flatten = nn.Flatten()
        self.head = nn.Sequential(
            nn.Linear(num_features, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 4),
        )


        # TODO 4: If `freeze` is True, set requires_grad = False for all backbone parameters.
        if freeze:
            for param in self.backbone.parameters():
                param.requires_grad_(False)


    def forward(self, x):
        # TODO 1: Pass x through backbone -> pool -> head and return the 4-d output.
        x = self.backbone(x)
        x = self.pool(x)
        x = self.flatten(x)
        x = self.head(x)
        return x



In [9]:
# Dataset & loader
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])
dataset = PennFudanDataset(root='data/PennFudanPed', transforms=transform)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

# Model & optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
backbone_model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model = ResNetBoxes(backbone_model, freeze=True).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# TODO 2: Choose an appropriate loss function for bounding box regression.
# compare e.g. nn.SmoothL1Loss() and nn.MSELoss()
criterion = nn.SmoothL1Loss()       #(Huber loss) is the standard choice for bounding box regression for two reasons: Outlier robustness — for large errors it grows linearly (like L1) rather than quadratically (like MSE), so a few badly predicted boxes don't dominate the gradient and destabilize training; Smooth gradient near zero — unlike pure L1, it doesn't have a discontinuous gradient at 0, which makes convergence near the optimum cleaner
#criterion = nn.MSELoss()       #MSELoss works but punishes large box errors so harshly that early training (when predictions are far off) can produce huge, destabilizing gradients — especially problematic when the backbone is frozen and only the small head is learning from scratch.

Intersection over Union (IoU) measures how well a predicted box overlaps with the ground truth:

$$\text{IoU} = \frac{\text{Area of Overlap}}{\text{Area of Union}}$$

Each box is represented as `[xmin, ymin, xmax, ymax]`.

In [11]:
def compute_iou(box1, box2):
    """Compute IoU between two boxes [xmin, ymin, xmax, ymax]."""
    # TODO 3: Implement this function.
    # Steps:
    #   1. Compute the coordinates of the intersection rectangle.
    #   2. Compute its area (watch out for non-overlapping boxes).
    #   3. Compute the area of each box.
    #   4. Return intersection_area / (area1 + area2 - intersection_area).

    # 1. Intersection rectangle
    inter_xmin = max(box1[0], box2[0])
    inter_ymin = max(box1[1], box2[1])
    inter_xmax = min(box1[2], box2[2])
    inter_ymax = min(box1[3], box2[3])

    # 2. Intersection area (clamped to 0 for non-overlapping boxes)
    inter_w = max(0, inter_xmax - inter_xmin)
    inter_h = max(0, inter_ymax - inter_ymin)
    inter_area = inter_w * inter_h

    # 3. Individual box areas
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])

    # 4. IoU
    union_area = area1 + area2 - inter_area
    return inter_area / union_area if union_area > 0 else 0.0

In [12]:
# Training loop
for epoch in range(5):
    model.train()
    running_loss = 0.0
    for imgs, targets in loader:
        imgs = imgs.to(device)
        gt_boxes = torch.stack([t for t in targets['box']]).to(device)

        preds = model(imgs)
        loss = criterion(preds, gt_boxes)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    # TODO: add evaluation on test dataset using our 'compute_iou'

    print(f"Epoch {epoch+1}, Loss: {running_loss / len(loader):.4f}")

Epoch 1, Loss: 206.3859
Epoch 2, Loss: 201.9467
Epoch 3, Loss: 204.4356
Epoch 4, Loss: 200.0884
Epoch 5, Loss: 197.7620


In [15]:
# Dataset & loaders
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])
dataset = PennFudanDataset(root='data/PennFudanPed', transforms=transform)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Model & optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
backbone_model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model = ResNetBoxes(backbone_model, freeze=True).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.SmoothL1Loss()

# Training loop
for epoch in range(5):
    model.train()
    running_loss = 0.0
    for imgs, targets in loader:
        imgs = imgs.to(device)
        gt_boxes = torch.stack([t for t in targets['box']]).to(device)
        preds = model(imgs)
        loss = criterion(preds, gt_boxes)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    model.eval()
    iou_scores = []
    with torch.no_grad():
        for imgs, targets in test_loader:
            imgs = imgs.to(device)
            gt_boxes = torch.stack([t for t in targets['box']]).to(device)
            preds = model(imgs)
            for pred, gt in zip(preds.cpu(), gt_boxes.cpu()):
                iou = compute_iou(pred.tolist(), gt.tolist())
                iou_scores.append(iou)

    mean_iou = sum(iou_scores) / len(iou_scores)
    print(f"Epoch {epoch+1}, Loss: {running_loss / len(loader):.4f}, Mean IoU: {mean_iou:.4f}")

Epoch 1, Loss: 197.5445, Mean IoU: 0.0000
Epoch 2, Loss: 203.9365, Mean IoU: 0.0000
Epoch 3, Loss: 202.1580, Mean IoU: 0.0000
Epoch 4, Loss: 194.6306, Mean IoU: 0.0000
Epoch 5, Loss: 198.0186, Mean IoU: 0.0000


In [ ]:
# TODO 5: Experiment here.
# For example:
#   backbone_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
#   model = ResNetBoxes(backbone_model, freeze=False).to(device)
#   ... retrain and evaluate ...
# Also try toggling `freeze=True` vs `freeze=False` — how does freezing the backbone affect convergence and final performance?